# A tour of react-bio-viz in Jupyter

This notebook loads a protein alignment, explores it interactively, and links it to a distance
matrix and a phylogenetic tree computed right here in Python. Every widget is interactive — pan,
zoom, select, drag — and every piece of interactive state is an ordinary traitlet you can read,
write and observe from the kernel.

**Install**

```bash
pip install react-bio-viz
```

or, from a checkout of the repository: `pnpm install && pnpm build`, then
`pip install -e packages/python`. The widgets work in JupyterLab, Notebook 7, VS Code and Colab.

In [ ]:
import json
import traitlets
from ipywidgets import Output

from react_bio_viz import MSA, DistanceMatrix, GeneModel, PhyloTree, parse_newick, read_fasta, to_newick

## 1. An alignment

`read_fasta` returns the plain structure the widgets take: a list of `{"header", "sequence"}`
records. This is a 24-sequence alignment of AUX/IAA family proteins.

In [ ]:
records = read_fasta("data/aux_iaa.fasta")
print(f"{len(records)} sequences × {len(records[0]['sequence'])} columns")

Scroll to pan, **Ctrl/⌘-scroll** (or pinch) to zoom, drag the minimap. **Shift-drag** across the
alignment to select columns, click a label to select a row, drag a label to reorder.

In [ ]:
msa = MSA(msa=records, width=820, height=480, options={"tracks": ["conservation", "logo"]})
msa

## 2. State is traitlets

What you do in the widget lands in its traits — `viewport`, `selection`, `row_order` and
`panel_sizes` — so the kernel can read it at any time…

In [ ]:
msa.selection

…react to it as it happens (callbacks print into an `Output` widget, since they run outside any
cell)…

In [ ]:
log = Output()

@log.capture(clear_output=True)
def show_selection(change):
    selected = change["new"]
    print(f"{len(selected['columns'])} columns, {len(selected['rows'])} rows selected")

msa.observe(show_selection, names="selection")
log

…and drive it. Writing a trait updates the widget above:

In [ ]:
msa.viewport = {**msa.viewport, "x0": 100, "x1": 160}           # jump to columns 101–160
msa.options = {**msa.options, "colorStyle": "AA Zappo"}          # a different colour scheme
msa.selection = {"rows": [], "columns": list(range(120, 130))}  # select ten columns

## 3. Editing

The widget never changes its data itself: renames and removals are reported through callbacks, and
you apply them. Here we keep the alignment in `msa.msa` and apply each edit. Double-click a label to
rename it, hover a label for its ×, or select columns and press **Delete**.

In [ ]:
# A stable id per row, so a renamed row keeps its identity (selection and row order use it). New
# records rather than mutating the old ones: traitlets only syncs a value that compares different.
msa.msa = [{**r, "id": r["header"]} for r in msa.msa]

def rename(row_id, name):
    msa.msa = [{**r, "header": name} if r["id"] == row_id else r for r in msa.msa]

def remove_rows(row_ids):
    msa.msa = [r for r in msa.msa if r["id"] not in row_ids]

def remove_columns(columns):
    drop = set(columns)
    msa.msa = [{**r, "sequence": "".join(c for i, c in enumerate(r["sequence"]) if i not in drop)} for r in msa.msa]

msa.on_rename_row(rename)
msa.on_remove_rows(remove_rows)
msa.on_remove_columns(remove_columns)

## 4. Distances, computed in Python

Any analysis works — the widgets only visualize. A p-distance is the share of differing positions
among those where neither sequence has a gap.

In [ ]:
def p_distances(records):
    seqs = [r["sequence"] for r in records]
    def distance(a, b):
        pairs = [(x, y) for x, y in zip(a, b) if x != "-" and y != "-"]
        return sum(x != y for x, y in pairs) / len(pairs) if pairs else 0.0
    return [[distance(a, b) for b in seqs] for a in seqs]

labels = [r["header"] for r in records]
distances = p_distances(records)

matrix = DistanceMatrix(labels=labels, matrix=distances, width=820, height=520,
                        options={"showNumbers": False, "cellWidth": 20, "cellHeight": 20})
matrix

The alignment and the matrix take the same `row_order` — ids in display order — so linking the two
traits keeps them in step. Drag a label in either one:

In [ ]:
row_link = traitlets.link((msa, "row_order"), (matrix, "row_order"))

## 5. A tree

Neighbour-joining (Saitou & Nei, 1987) builds a tree from the distances. The result is the nested
`{"name", "length", "children"}` structure `PhyloTree` takes.

In [ ]:
def neighbor_joining(labels, distances):
    nodes = [{"name": name, "length": 0.0, "children": []} for name in labels]
    d = [list(row) for row in distances]
    while len(nodes) > 3:
        n = len(nodes)
        r = [sum(row) for row in d]
        i, j = min(((i, j) for i in range(n) for j in range(i + 1, n)),
                   key=lambda ij: (n - 2) * d[ij[0]][ij[1]] - r[ij[0]] - r[ij[1]])
        li = d[i][j] / 2 + (r[i] - r[j]) / (2 * (n - 2))
        joined = {"name": "", "length": 0.0, "children": [
            {**nodes[i], "length": max(0.0, li)},
            {**nodes[j], "length": max(0.0, d[i][j] - li)},
        ]}
        keep = [k for k in range(n) if k not in (i, j)]
        to_joined = [(d[i][k] + d[j][k] - d[i][j]) / 2 for k in keep]
        d = [[d[a][b] for b in keep] + [to_joined[x]] for x, a in enumerate(keep)] + [to_joined + [0.0]]
        nodes = [nodes[k] for k in keep] + [joined]
    a, b, c = range(len(nodes))
    lengths = [(d[a][b] + d[a][c] - d[b][c]) / 2, (d[a][b] + d[b][c] - d[a][c]) / 2, (d[a][c] + d[b][c] - d[a][b]) / 2]
    return {"name": "", "length": 0.0,
            "children": [{**node, "length": max(0.0, l)} for node, l in zip(nodes, lengths)]}

tree = neighbor_joining(labels, distances)

Drag a node to another row to rotate the tree, or past the top or bottom to reroot on it. Click an
internal node's marker to collapse it.

In [ ]:
phylo = PhyloTree(tree=tree, width=820, height=480, leaf_spacing=18, interactive=True,
                  show_support_values=False)
phylo

`leaf_order` reports the tree's leaves in display order. Feed it to the alignment (and, through the
link above, the matrix), and the rows follow the tree — as in a phylogenetic MSA viewer:

In [ ]:
tree_link = traitlets.dlink((phylo, "leaf_order"), (msa, "row_order"))

Clicks arrive through callbacks with everything about the node — its leaves, all descendant ids,
and a ready-made selection to reroot on it:

In [ ]:
clicks = Output()

@clicks.capture(clear_output=True)
def on_node(node):
    kind = "leaf" if node["isLeaf"] else f"clade of {len(node['leafNames'])}"
    print(f"{node['id']}: {kind} — {', '.join(node['leafNames'][:5])}{' …' if len(node['leafNames']) > 5 else ''}")

phylo.on_node_click(on_node)
clicks

The arrangement is state too — root placement, collapsed clades, child order — so it can be set
from Python. Collapse the clade you last clicked, or reset:

In [ ]:
phylo.selection

In [ ]:
phylo.selection = {"collapsed": []}  # back to the tree as computed

## 6. Newick

`parse_newick` and `to_newick` convert, handling quoted labels:

In [ ]:
newick = to_newick(tree)
print(newick[:120], "…")
assert parse_newick(newick)["children"][0]["name"] == tree["children"][0]["name"]

## 7. Genes

The genome widgets work the same way. A gene model, with its transcripts, exons and CDSs — click an
exon for its details:

In [ ]:
with open("data/genemodel.json") as handle:
    gene = json.load(handle)

GeneModel(gene=gene, width=820)

## More

`GenomeBrowser` stacks feature, coverage and gene tracks; `BlastHitDistribution` shows BLAST hits
along a query. See the [documentation](https://holmrenser.github.io/react-bio-viz/) for every widget,
trait and option.